# Crypto multi-strategy book — five low-correlation sleeves

**Slug:** `crypto-multi-strategy-book-v3` · **Status:** `accept_monitoring` · **Date:** 2026-06-04

Five sleeves, each anchored to a **different return source** (carry · trend · cross-sectional momentum ·
flow/forced-trade · macro-regime), so PnL is orthogonal by construction, then combined. All daily,
leak-safe, **excess of cash**. Window 2022–2025 (2025 OOS; 2026 PM-holdout excluded).

- Decision record: [`docs/research_decisions/crypto_intraday/P7-multi-strategy-book.md`](../../docs/research_decisions/crypto_intraday/P7-multi-strategy-book.md)
- Report: `reports/crypto_v3_multi_strategy.html` (render with `render_multi_strategy_report.py`)

Ideas came from the `idea-generation` skill (random stimuli → operators → one idea per return source).

In [ ]:
import sys, json
from pathlib import Path
# find repo root from the notebook cwd (no hardcoded path)
ROOT = Path.cwd()
while not (ROOT / "src" / "alpha_lab").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd
from alpha_lab.data.loaders.binance_vision import load_klines, load_funding
from alpha_lab.data.loaders.fred import load_series, discount_rate_to_daily_rate
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.backtest.metrics import summary
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 30)

START, END, BARS = "2022-01-01", "2026-01-01", 365
BURN = pd.Timestamp("2022-04-01", tz="UTC")
OUT = ROOT / "data" / "results" / "crypto_v3_multi"; OUT.mkdir(parents=True, exist_ok=True)
SLIP = {"BTC.p": 8., "ETH.p": 10., "SOL.p": 12., "BNB.p": 12., "BTC.s": 15., "ETH.s": 17.5}
SYM = {"BTCUSDT": "BTC", "ETHUSDT": "ETH", "SOLUSDT": "SOL", "BNBUSDT": "BNB"}
RF_FALLBACK = {2022: 0.020, 2023: 0.0521, 2024: 0.0505, 2025: 0.0425}

def banded(z, en, ex):
    """Hysteresis position in {-1,0,+1} for a reversion signal (cuts turnover)."""
    raw = pd.Series(np.where(z >= en, -1., np.where(z <= -en, 1., np.where(z.abs() <= ex, 0., np.nan))), index=z.index)
    return raw.ffill().fillna(0.)

## 1 · Data — daily klines (spot + perp) + 8h funding

In [ ]:
perp = load_klines(list(SYM), "1d", START, END, market="perp").close_panel().rename(columns={k: f"{v}.p" for k, v in SYM.items()})
spot = load_klines(["BTCUSDT", "ETHUSDT"], "1d", START, END, market="spot").close_panel().rename(columns={"BTCUSDT": "BTC.s", "ETHUSDT": "ETH.s"})
funding = load_funding(list(SYM), START, END).rename(columns={k: f"{SYM[k]}.p" for k in SYM})
grid = perp.index.union(spot.index)
perp, spot = perp.reindex(grid).ffill(), spot.reindex(grid).ffill()
prices = pd.concat([spot, perp], axis=1)
print("grid", grid.min().date(), "->", grid.max().date(), "n=", len(grid))

## 2 · Cost of cash (rf) + macro gate input (beyond price-volume)
rf = 3M T-bill financing (FRED DTB3, fallback piecewise). Macro = HYG credit ETF trend.

In [ ]:
rf_src = None
try:
    rf_src = discount_rate_to_daily_rate(load_series("DTB3", start=START, end=END, timeout=25)["DTB3"].dropna())
    print("rf: FRED DTB3")
except Exception as e:
    print("rf: fallback (FRED failed:", type(e).__name__, ")")

def align_rf(idx):
    naive = idx.tz_localize(None).normalize()
    if rf_src is not None:
        r = rf_src.copy(); r.index = pd.DatetimeIndex(r.index).normalize()
        out = r.reindex(naive).ffill().fillna(0.0)   # trailing ffill only (leak-safe); leading edge -> 0
    else:
        out = pd.Series([RF_FALLBACK.get(d.year, .04) / 365 for d in naive], index=naive)
    out.index = idx; return out.astype(float)

rf_daily = align_rf(grid)
hyg = load_prices("HYG", START, END)["HYG"]; hyg.index = pd.DatetimeIndex(hyg.index).tz_localize("UTC")
hyg = hyg.reindex(grid).ffill()
print("macro: HYG credit ETF 50d trend; rf ~", round(float(rf_daily.mean())*365*100, 2), "% ann")

## 3 · Helpers — backtest, cost-of-cash excess, daily funding
`r_excess = r_raw − rf · gross_long_notional` (uniform cost-of-cash); engine lags weights 1 bar.

In [ ]:
def bt(w, cols, use_funding):
    w = w.reindex(columns=cols).reindex(grid).fillna(0.)
    fund = funding[[c for c in cols if c in funding.columns]] if use_funding else None
    return run_backtest(signals=w, prices=prices[cols], rebalance=None, costs_bps=0.,
                        slippage_bps={c: SLIP[c] for c in cols}, funding=fund, bars_per_year=BARS)

def exc(res):
    L = res.weights.clip(lower=0).sum(axis=1)
    return (res.returns - rf_daily.reindex(L.index).fillna(0) * L).rename("r")

f = funding.copy(); f.index = pd.DatetimeIndex(f.index).tz_convert("UTC")
df_fund = f.resample("1D").mean(); df_fund.index = pd.DatetimeIndex(df_fund.index)
if df_fund.index.tz is None: df_fund.index = df_fund.index.tz_localize("UTC")
df_fund = df_fund.reindex(grid).ffill()

sleeves, diag = {}, {}
def record(name, res, source):
    sleeves[name] = exc(res)
    diag[name] = {"source": source,
                  "gross_sharpe": summary(res.gross_returns.loc[BURN:], periods=BARS).get("Sharpe", np.nan),
                  "ann_turnover": float(res.turnover.loc[BURN:].sum()) / (len(res.turnover.loc[BURN:]) / BARS),
                  "time_in_mkt": float((res.weights.abs().sum(axis=1).loc[BURN:] > 1e-9).mean())}
    print(f"{name:14s} netSh={summary(sleeves[name].loc[BURN:], periods=BARS)['Sharpe']:.2f}")

## 4 · The five sleeves
**S1 Carry** — long spot / short perp when 7d funding > 0 (market-neutral; the P6 edge).

In [ ]:
act = (df_fund.rolling(7).mean() > 0).astype(float)
w1 = pd.DataFrame(0., index=grid, columns=["BTC.s", "ETH.s", "BTC.p", "ETH.p"])
for c in ["BTC", "ETH"]: w1[f"{c}.s"], w1[f"{c}.p"] = .25 * act[f"{c}.p"], -.25 * act[f"{c}.p"]
record("S1_carry", bt(w1, ["BTC.s", "ETH.s", "BTC.p", "ETH.p"], True), "perp funding carry (market-neutral)")

**S2 Trend** — BTC/ETH perp long above / short below 50d MA (TS momentum, long/short).

In [ ]:
sma = perp[["BTC.p", "ETH.p"]].rolling(50).mean()
w2 = 0.5 * (2 * (perp[["BTC.p", "ETH.p"]] > sma).astype(float) - 1)
record("S2_trend", bt(w2, ["BTC.p", "ETH.p"], True), "time-series momentum (long/short)")

**S3 XS-Mom** — long top2 / short bot2 of {BTC,ETH,SOL,BNB} by 30d return (market-neutral).

In [ ]:
ret30 = perp / perp.shift(30) - 1
rk = ret30.rank(axis=1)
w3 = (rk.sub(rk.mean(axis=1), axis=0)) / 3.0
record("S3_xsmom", bt(w3, ["BTC.p", "ETH.p", "SOL.p", "BNB.p"], True), "cross-sectional momentum (market-neutral)")

**S4 Funding-contra** — banded fade of funding z-extremes (flow / forced-trade; dormant until stretched).

In [ ]:
w4 = pd.DataFrame(0., index=grid, columns=["BTC.p", "ETH.p"])
for c in ["BTC.p", "ETH.p"]:
    z = (df_fund[c] - df_fund[c].rolling(30).mean()) / df_fund[c].rolling(30).std()
    w4[c] = 0.5 * banded(z, 1.0, 0.3)
record("S4_fundcontra", bt(w4, ["BTC.p", "ETH.p"], True), "funding-extreme contrarian (flow / forced-trade)")

**S5 Macro-gate** — hold crypto only when HYG credit regime is risk-on (beyond price-volume).

In [ ]:
ro = (hyg.shift(1) > hyg.shift(1).rolling(50).mean()).astype(float)
w5 = pd.DataFrame({"BTC.s": .5 * ro, "ETH.s": .5 * ro})
record("S5_macro", bt(w5, ["BTC.s", "ETH.s"], False), "macro credit-regime gate (beyond price-volume)")

## 5 · Correlation — orthogonal by design

In [ ]:
R = pd.DataFrame(sleeves).loc[BURN:].dropna(how="all")
corr = R.corr()
offdiag = corr.where(~np.eye(len(corr), dtype=bool)).abs()
print("mean|offdiag|=", round(float(offdiag.mean().mean()), 3), " max=", round(float(offdiag.max().max()), 3),
      " sum_pairs=", round(float(corr.where(np.triu(np.ones_like(corr, bool), 1)).sum().sum()), 3))
corr.round(2)

In [ ]:
tbl = pd.DataFrame({k: summary(R[k].dropna(), periods=BARS) for k in R.columns}).T
tbl["GrossSharpe"] = [diag[k]["gross_sharpe"] for k in tbl.index]
tbl["AnnTO"] = [diag[k]["ann_turnover"] for k in tbl.index]
tbl["TiM"] = [diag[k]["time_in_mkt"] for k in tbl.index]
tbl[["CAGR", "AnnVol", "Sharpe", "GrossSharpe", "MaxDD", "Calmar", "AnnTO", "TiM"]].round(3)

## 6 · Combination — equal-capital + risk-budget (the 'control room')

In [ ]:
TARGET_EACH, LCAP = 0.08, 10.0
svol = (R.rolling(60).std().shift(1) * np.sqrt(BARS))
lev = (TARGET_EACH / svol).clip(upper=LCAP).fillna(0.)
combo_eqcap = R.mean(axis=1).rename("combo_eqcap")
combo_rb = (lev * R).mean(axis=1).rename("combo_riskbudget")
rv = combo_rb.rolling(60).std().shift(1) * np.sqrt(BARS)
combo_vt = ((0.10 / rv).clip(upper=3.0).fillna(0.) * combo_rb).rename("combo_vt10")
btc_bh = (spot["BTC.s"].pct_change() - rf_daily).reindex(R.index).rename("btc_bh_excess")
DR = float((pd.Series(0.2, index=R.columns) * R.std()).sum() / combo_eqcap.std())

books = {"equal-capital (20% ea)": combo_eqcap, "risk-budget (cap10x)": combo_rb,
         "risk-budget vt10%": combo_vt, "BTC buy&hold (excess)": btc_bh}
print("diversification ratio =", round(DR, 2), "| carry leverage ~", round(float(lev.loc[BURN:].mean()['S1_carry']), 1), "x")
pd.DataFrame({nm: summary(s.dropna(), periods=BARS) for nm, s in books.items()}).T[
    ["CAGR", "AnnVol", "Sharpe", "Sortino", "MaxDD", "Calmar"]].round(3)

In [ ]:
# by year (equal-capital combo vs BTC)
rows = []
for y in [2022, 2023, 2024, 2025]:
    e = (1 + combo_eqcap[combo_eqcap.index.year == y].fillna(0)).prod() - 1
    b = (1 + btc_bh[btc_bh.index.year == y].fillna(0)).prod() - 1
    rows.append({"year": y, "combo": e, "BTC": b})
pd.DataFrame(rows).set_index("year").round(3)

## 7 · Persist (for the HTML report step)

In [ ]:
R.to_parquet(OUT / "sleeve_excess_returns.parquet")
pd.DataFrame({"combo_eqcap": combo_eqcap, "combo_riskbudget": combo_rb,
              "combo_vt10": combo_vt, "btc_bh_excess": btc_bh}).to_parquet(OUT / "combos.parquet")
corr.to_parquet(OUT / "corr.parquet"); tbl.to_parquet(OUT / "sleeve_summary.parquet")
lev.to_parquet(OUT / "rb_leverage.parquet")
meta = {"mean_offdiag_corr": float(offdiag.mean().mean()), "max_offdiag_corr": float(offdiag.max().max()),
        "sum_pairwise_corr": float(corr.where(np.triu(np.ones_like(corr, bool), 1)).sum().sum()),
        "diversification_ratio": DR, "rf_source": "FRED DTB3" if rf_src is not None else "fallback piecewise",
        "macro_source": "yfinance HYG (high-yield credit ETF) 50d trend", "target_each": TARGET_EACH, "lcap": LCAP,
        "mean_leverage": {k: float(lev.loc[BURN:].mean()[k]) for k in R.columns}, "diag": diag}
(OUT / "meta.json").write_text(json.dumps(meta, indent=2, default=float))
print("saved ->", OUT)
print("render the HTML report with:  python render_multi_strategy_report.py")

## 8 · Decision

**`accept_monitoring`** as a diversified book (not a single alpha). Mean |ρ| ≈ 0.11, diversification
ratio ≈ 2.0; combined Sharpe ≈ 1.15 vs BTC 0.51, CAGR 20% vs 14%, MaxDD −15% vs −67%; positive every
calendar year incl. the 2025 OOS. The edge is the **low correlation** itself. Monitor the carry
funding−financing spread, the rolling S2↔S3 correlation, and the trailing-quarter combined Sharpe.
See the decision record for full robustness, failure modes, and next steps.